In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import pandas as pd
import os
import torch
import matplotlib
#from sklearn.utils.sparsefuncs import mean_variance_axis
#import gseapy as gp
#from sklearn.preprocessing import MinMaxScaler

In [ ]:
adata = sc.read_h5ad('data/spatial/processed_data/scRNA.h5ad')

gene_list = (
    pd.read_csv("data/spatial/genes/BCLL-9-T.csv")["gene"]
    .dropna()
    .astype(str)
    .tolist()
)
gene_list = [g for g in gene_list if g in adata.var_names]
adata = adata[:, gene_list].copy()
adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

sc.pp.log1p(adata)
sc.tl.rank_genes_groups(
    adata,
    groupby='cell_type',
    use_raw=False,
    pts=True,
    method='wilcoxon'
)
'''
sc.pl.rank_genes_groups_dotplot(adata, groupby="annotation_level_1", standard_scale="var", n_genes=5)

groups = adata.uns['rank_genes_groups']['names'].dtype.names 
dfs = []

for group in groups:
    df = pd.DataFrame({
        'gene': adata.uns['rank_genes_groups']['names'][group],
        'logfoldchange': adata.uns['rank_genes_groups']['logfoldchanges'][group],
        'pvals': adata.uns['rank_genes_groups']['pvals'][group],
        'pvals_adj': adata.uns['rank_genes_groups']['pvals_adj'][group],
        'scores': adata.uns['rank_genes_groups']['scores'][group]
    })
    df['group'] = group
    dfs.append(df)
result_df = pd.concat(dfs, axis=0)
result_df.to_csv('data/spatial/markers/markers.csv', index=False)
'''

In [ ]:
def run_selected_genes_gsea(
    adata_sc,
    groupby: str,
    selected_genes_spec: pd.DataFrame,
    rank_stat: str = "scores", 
    topn:int = 1000,
    min_size: int = 15,
    max_size: int = 2000,
    permutation_num: int = 1000,
    seed: int = 0,
):
    """
    for each cell type ct:
      1) from scanpy's rank_genes_groups to get the gene ranking list for ct (based on rank_stat)
      2) from selected_genes_spec get the "selected gene set" for ct
      3) do preranked GSEA to test if this gene set is enriched at the top of the ranking
    Returns: summary DataFrame + detailed results dictionary for each ct
    """

    # 1) make sure rank_genes_groups has been run
    if "rank_genes_groups" not in adata_sc.uns:
        raise ValueError("adata_sc.uns['rank_genes_groups'] not found. "
                         "Run sc.tl.rank_genes_groups(...) first.")

    # 2) cell type list: take intersection to avoid mismatch in column names of selected_genes_spec
    cts_sc = pd.Index(adata_sc.obs[groupby].unique().tolist())
    cts_sel = pd.Index(selected_genes_spec.columns.tolist())
    cts = [ct for ct in cts_sc if ct in cts_sel]

    if len(cts) == 0:
        raise ValueError("No overlapping cell types between adata_sc.obs[groupby] and selected_genes_spec.columns")

    summary_rows = []
    detailed = {}

    for ct in cts:
        # ---  DE rank  ---
        df = sc.get.rank_genes_groups_df(adata_sc, group=ct)

        if rank_stat not in df.columns:
            raise KeyError(f"{rank_stat} not in rank_genes_groups_df columns. Available: {df.columns.tolist()}")

        # df usually has been ranked by score but we sort again to be safe
        rnk = (
            df[["names", rank_stat]]
            .dropna()
            .drop_duplicates(subset=["names"])
            .set_index("names")[rank_stat]
            .astype(float)
        )

        rnk = rnk.loc[rnk.index.intersection(adata_sc.var_names)]
        rnk = rnk.sort_values(ascending=False)

        # ---  selected genes set (1000 genes) ---
        col = selected_genes_spec[ct]

        # if col.dtype == bool:
        #     sel_genes = col.index[col].tolist()
        # else:
        #     sel_genes = col.index[col.notna() & (col.values != 0)].tolist()

        # intersect with rank list
        sel_genes = list(set(col[0:topn]) & set(rnk.index))

        if len(sel_genes) < min_size:
            summary_rows.append({
                "ct": ct,
                "n_selected_in_rank": len(sel_genes),
                "NES": np.nan,
                "pval": np.nan,
                "FDR": np.nan,
                "note": f"skip: selected genes < min_size ({min_size}) after intersection"
            })
            continue

        gene_sets = {"selected_genes": sel_genes}

        # gseapy.prerank needs two columns: gene and rank metric
        rnk_df = rnk.reset_index()
        rnk_df.columns = ["gene", "score"]

        pre_res = gp.prerank(
            rnk=rnk_df,
            gene_sets=gene_sets,
            min_size=min_size,
            max_size=max_size,
            permutation_num=permutation_num,
            seed=seed,
            outdir=None,          # don't write files
            no_plot=True,
            verbose=False,
        )

        res = pre_res.res2d # one gene set
        detailed[ct] = pre_res

        summary_rows.append({
            "ct": ct,
            "n_selected_in_rank": len(sel_genes),
            "NES": float(res["NES"]),
            "pval": float(res["NOM p-val"]),
            "FDR": float(res["FDR q-val"]),
           # "Gene": str(res["Gene %"]),
            "note": ""
        })

    summary = pd.DataFrame(summary_rows).sort_values(["FDR", "pval"], ascending=True)
    return summary, detailed


In [ ]:
folder_path = "data/spatial/R_score/BCLL-9-T"

genes_dict = {}
scaler = MinMaxScaler()

for file in os.listdir(folder_path):
    if file.endswith(".csv") or file.endswith(".txt"):
        ct_name = os.path.splitext(file)[0] 
        
        file_path = os.path.join(folder_path, file)

        df = pd.read_csv(file_path, header=0)  
        df.columns = df.columns.str.strip()    
        #df = df[(df['delta_R2'] > 0)]
        df = df[(df['R2_1'] > 0)]#.sort_values(by=['delta_R2','R4'], ascending=False)
        df = df[(df['R4'] > 0)]
        #df[['delta_R2', 'R4']] = scaler.fit_transform(df[['delta_R2', 'R4']])

        #df['combined_score'] = 0.5*df['delta_R2'] + 0.5*df['R4']
        df = df.sort_values(by=['R4'], ascending=False)
        genes_dict[ct_name] = df['gene'].tolist()
        
selected_genes_spec = pd.DataFrame({k: pd.Series(v) for k, v in genes_dict.items()})

In [ ]:
summary, detailed = run_selected_genes_gsea(
        adata,
        groupby="cell_type",
        selected_genes_spec=selected_genes_spec,
        rank_stat="scores", 
        topn=80,
    )
summary

In [ ]:
summary, detailed = run_selected_genes_gsea(
        adata,
        groupby="cell_type",
        selected_genes_spec=selected_genes_spec,
        rank_stat="scores", 
        topn=100,
    )
summary

In [ ]:
selected_genes_spec[:200].to_csv('data/spatial/selected_genes/BCLL-9-T.csv')

In [ ]:
top_genes_df = selected_genes_spec[:50]

cts = top_genes_df.columns.tolist()
n = len(cts)

intersection_matrix = pd.DataFrame(
    np.zeros((n, n), dtype=int),
    index=cts,
    columns=cts
)

for ct1 in cts:
    genes1 = set(top_genes_df[ct1].dropna())
    for ct2 in cts:
        genes2 = set(top_genes_df[ct2].dropna())
        intersection_matrix.loc[ct1, ct2] = len(genes1 & genes2)

intersection_matrix

In [ ]:
summary, detailed = run_selected_genes_gsea(
        adata,
        groupby="cell_type",
        selected_genes_spec=selected_genes_spec,
        rank_stat="scores", 
        topn=150,
    )
summary

In [ ]:
adata.write("adata_marker.h5ad")

In [ ]:
adata = sc.read_h5ad('adata_marker.h5ad')
marker_genes = {}

for cell_type in adata.obs['cell_type'].unique():

    result = sc.get.rank_genes_groups_df(adata, group=cell_type)
    
    significant = result[result['pvals_adj'] < 0.05]
    
    top500 = significant.nlargest(2000, 'scores')
    marker_genes[cell_type] = top500['names'].tolist()
    
    print(f"{cell_type}: {len(marker_genes[cell_type])} genes")

df = pd.DataFrame.from_dict(marker_genes, orient='index').T
df

In [ ]:
df.to_csv('marker.csv', index=False)